In [1]:
from tqdm import tqdm
import pandas as pd
import spacy
import re

In [ ]:
pd.set_option('display.max_columns', 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [4]:
presse = pd.read_excel("/Users/pol/Downloads/press_corpus.xlsx")

presse = presse.dropna(subset=["article_text"])

In [5]:
presse = presse.rename(columns={"id": "article_id"})
presse

,article_id,source,article_text
0,1,telerama,Serge Reggiani - Succès & confidences .\n Une ...
1,2,telerama,"Désolés, Warren .\n C'est une maigre consolati..."
2,3,telerama,Adam Green - Friends of mine .\n Dire qu'on a ...
3,4,telerama,Cesaria Evora - Voz d'amor .\n Dès les premièr...
4,5,telerama,Herman Düne - Mas cambios et Mash Concrete Met...
...,...,...,...
52689,52690,lemonde,"Rap, jazz, rock, classique, chanson… Les album..."
52690,52691,lemonde,Que sait-on vraiment des surrisques de contami...
52691,52692,lemonde,"Rencontre avec Riccardo Muti, maestro à vie .\..."
52692,52693,lemonde,Ils sont passés chez Colors : la sélection mus...


# General NER retrieval

In [6]:
nlp = spacy.load('fr_core_news_lg')

In [7]:
df = presse

# create spacy doc object with article_text, keep article id
# parallelize
docs = nlp.pipe(
    zip(df["article_text"], df["article_id"]),
    as_tuples=True,
    disable=['tok2vec', 'parser', 'lemmatizer'],
    batch_size=8,
    n_process=4
)

result = []

# for all ents that are not LOC, extract ents contained in each article
with tqdm(total=len(df)) as pbar:
    for doc, idx in docs: 
        for ent in doc.ents:
            if  ent.label_ != 'LOC':
                result.append((idx, ent.text, ent.label_))
            
        pbar.update(1)
        
ner_df = pd.DataFrame(result, columns=['article_id', 'name', 'ner_type'])

100%|██████████| 52694/52694 [36:29<00:00, 24.07it/s]  


In [16]:
# count occurrences of each name
ner_df["name_count"] = ner_df["name"].map(ner_df["name"].value_counts())
ner_df

,article_id,name,ner_type,name_count
0,1,Serge Reggiani - Succès & confidences,PER,1
1,1,Reggiani,ORG,36
2,1,Brel,PER,337
3,1,Barbara,PER,663
4,1,Ventura,PER,16
...,...,...,...,...
1472688,52694,Moussorgsky,PER,4
1472689,52694,Galina Vichnevskaïa,PER,10
1472690,52694,Arturo Benedetti Michelangeli,PER,9
1472691,52694,Ballade no 1,MISC,2


In [107]:
# df of unique extracted entities
extracted_ents = (
    ner_df
    .groupby("name")
    .agg({"article_id": "first",
          "name_count": "first"})
    .sort_values(by='name_count', ascending=False)
    .reset_index()
)

In [108]:
# solve tiny issue with names starting with - (that lead to csv encoding error)
def remove_tiretdusix(string):
    string = re.sub(pattern="^-", repl="", string=string)
    string = string.strip()
    return string

extracted_ents["name"] = extracted_ents["name"].apply(remove_tiretdusix)

In [109]:
# reset index and reorder cols
extracted_ents.reset_index()
extracted_ents.index += 1 # 1-based index
extracted_ents.index.name = "name_id"

# careful: in this df, article_id displays the article with the first occurrence of a name
extracted_ents = extracted_ents[["name", "name_count", "article_id"]]
extracted_ents

,name,name_count,article_id
name_id,,,
1,Ça,4723,9
2,CD,3801,2
3,Libération,3336,870
4,Figaro,2749,1786
5,Internet,2376,40
...,...,...,...
408674,I’m Amazing,1,49542
408675,I’m All Smiles,1,50227
408676,I’ll Stand by You,1,51271


In [110]:
extracted_ents.to_csv("/Users/pol/Downloads/extracted_ents_ALL.csv", 
                      sep=";",
                      encoding="utf-8-sig")

In [45]:
# press_corpus with column listing all entities for each text
article_text = presse[["article_id", "article_text"]]
article_text.index += 1 # 1-based index

article_text

,article_id,article_text
1,1,Serge Reggiani - Succès & confidences .\n Une ...
2,2,"Désolés, Warren .\n C'est une maigre consolati..."
3,3,Adam Green - Friends of mine .\n Dire qu'on a ...
4,4,Cesaria Evora - Voz d'amor .\n Dès les premièr...
5,5,Herman Düne - Mas cambios et Mash Concrete Met...
...,...,...
52690,52690,"Rap, jazz, rock, classique, chanson… Les album..."
52691,52691,Que sait-on vraiment des surrisques de contami...
52692,52692,"Rencontre avec Riccardo Muti, maestro à vie .\..."
52693,52693,Ils sont passés chez Colors : la sélection mus...


In [105]:
agg_matches = (
    ner_df
    .groupby("article_id").agg({
            "name": list,
            "name_count": "first",
        })
    .join(article_text, on="article_id")    
)

agg_matches = agg_matches[["article_text", "name", "name_count"]]
agg_matches

,article_text,name,name_count
article_id,,,
1,Serge Reggiani - Succès & confidences .\n Une ...,"[Serge Reggiani - Succès & confidences, Reggia...",1
2,"Désolés, Warren .\n C'est une maigre consolati...","[Warren, Warren Zevon, I'll sleep when I'm dea...",10
3,Adam Green - Friends of mine .\n Dire qu'on a ...,"[Adam Green - Friends of mine, The Kills, Gree...",1
4,Cesaria Evora - Voz d'amor .\n Dès les premièr...,"[gorgée, Voz, Voix de l'amour, Velocidade, Ces...",2
5,Herman Düne - Mas cambios et Mash Concrete Met...,"[Herman Düne - Mas cambios, Mash Concrete Meta...",1
...,...,...,...
52690,"Rap, jazz, rock, classique, chanson… Les album...","[Rap, Monde, An Evening with Silk Sonic, Silk ...",45
52691,Que sait-on vraiment des surrisques de contami...,"[Covid-19, Omicron, Jean Castex, Covid-19, Clu...",482
52692,"Rencontre avec Riccardo Muti, maestro à vie .\...","[Riccardo Muti, Riccardo Muti, Cristina, Ricca...",70


In [106]:
agg_matches.to_csv("/Users/pol/Downloads/agg_matches_ALL.csv", 
                   sep=";",
                   encoding="utf-8-sig")
agg_matches

,article_text,name,name_count
article_id,,,
1,Serge Reggiani - Succès & confidences .\n Une ...,"[Serge Reggiani - Succès & confidences, Reggia...",1
2,"Désolés, Warren .\n C'est une maigre consolati...","[Warren, Warren Zevon, I'll sleep when I'm dea...",10
3,Adam Green - Friends of mine .\n Dire qu'on a ...,"[Adam Green - Friends of mine, The Kills, Gree...",1
4,Cesaria Evora - Voz d'amor .\n Dès les premièr...,"[gorgée, Voz, Voix de l'amour, Velocidade, Ces...",2
5,Herman Düne - Mas cambios et Mash Concrete Met...,"[Herman Düne - Mas cambios, Mash Concrete Meta...",1
...,...,...,...
52690,"Rap, jazz, rock, classique, chanson… Les album...","[Rap, Monde, An Evening with Silk Sonic, Silk ...",45
52691,Que sait-on vraiment des surrisques de contami...,"[Covid-19, Omicron, Jean Castex, Covid-19, Clu...",482
52692,"Rencontre avec Riccardo Muti, maestro à vie .\...","[Riccardo Muti, Riccardo Muti, Cristina, Ricca...",70
